# Demo: recovering apple-tree leader structure from one summer RGB-D image

MMA3001 project, Shreeji Kumawat. This notebook is the live demonstration used in the
presentation. It loads one **test-row** tree (never seen in training), runs the classical
baseline and the trained U-Net, and reports the structural quantities a pruning robot needs.

Run from the repository root after following `data/README.md`.

In [ ]:
import sys, json
sys.path.insert(0, 'src')
import numpy as np, torch, matplotlib.pyplot as plt
from treeid.io import discover, load_sample
from treeid.preprocess import make_input, crop_resize, foliage_mask
from treeid.model import UNet
from treeid import baseline
from treeid.metrics import score_all, shift_compensated_f1
from treeid.skeleton import leader_angle, backproject
torch.set_num_threads(1)

## 1. Load one tree from a held-out test row

In [ ]:
recs, _ = discover('data/raw')
TREE = 'R10N05'                       # row 10 is a test row
s = load_sample(recs[TREE])
gt = s['gt']
print(TREE, 'RGB', s['rgb'].shape, 'depth', s['depth'].shape, 'GT', gt.shape,
      f"missing depth {(s['depth']==0).mean():.0%}")

## 2. Classical baseline (ridge filter + depth + vegetation index)

In [ ]:
x256 = make_input(s['rgb'], s['depth'], 256, near_is_bright=False)
params = baseline.BaselineParams(**json.load(open('results/runs/baseline/summary.json'))['params'])
pred_b = baseline.predict(x256[:3].transpose(1, 2, 0), x256[3], x256[4] > 0.5, params)

## 3. U-Net (RGB-D input, 128 px)

In [ ]:
summ = json.load(open('results/runs/rgbd_s0/summary.json'))
model = UNet(5); model.load_state_dict(torch.load('results/runs/rgbd_s0/model.pt')); model.eval()
x128 = make_input(s['rgb'], s['depth'], 128, near_is_bright=False)
import time, cv2
t0 = time.perf_counter()
with torch.no_grad():
    prob = torch.sigmoid(model(torch.from_numpy(x128)[None]))[0, 0].numpy()
print(f'inference {1000*(time.perf_counter()-t0):.1f} ms on 1 CPU thread')
pred_u = cv2.resize(prob, (256, 256)) > summ['threshold']

## 4. Scores against the hand-labelled mask

In [ ]:
occ = foliage_mask(crop_resize(s['rgb'], 256))
for name, p in [('baseline', pred_b), ('U-Net', pred_u)]:
    m = score_all(p, gt, occ, tol=4)
    comp, k = shift_compensated_f1(p, gt, tol=4)
    print(f"{name:9s} centreline F1 {m['cl_f1']:.2f} | shift-compensated {comp:.2f} (label offset {k:+d} px)"
          f" | occluded recall {m['occ_recall']:.2f} | IoU {m['iou']:.2f}")

## 5. Structure for the robot: fork height, leader angle, 3-D skeleton

In [ ]:
for name, m in [('GT', gt), ('U-Net', pred_u)]:
    a, info = leader_angle(m)
    print(f"{name:6s} leader angle {a:.1f} deg, fork at row {info['fork_row']} of 256")
d256 = crop_resize(s['depth'], 256, is_mask=True)
pts = backproject(pred_u, d256.astype(float), fx=328, fy=328, cx=128, cy=128)   # intrinsics: assumed, see report
print(pts.shape[0], 'skeleton points with depth; Z range (raw units):', pts[:,2].min(), pts[:,2].max())

In [ ]:
rgb = crop_resize(s['rgb'], 256)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(rgb); ax[0].set_title('summer RGB (crop)')
ax[1].imshow(rgb); ax[1].contour(gt, [.5], colors='r'); ax[1].set_title('hand label (red)')
for a_, p, t in [(ax[2], pred_b, 'baseline'), (ax[3], pred_u, 'U-Net')]:
    a_.imshow(rgb); a_.imshow(np.ma.masked_where(~p, p), cmap='cool', alpha=.6); a_.contour(gt, [.5], colors='r'); a_.set_title(t)
for a_ in ax: a_.axis('off')
plt.tight_layout()